# 05 - Métodos de ponto fixo
Vamos aprender sobre como usar os métodos de ponto fixo para encontrar as raízes de funções reais.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana5
```

Faça o checkout nessa nova branch:

```bash
git checkout semana5
```

Instale as bibliotecas NumPy e SciPy.

In [2]:
%pip install numpy scipy

  Obtaining dependency information for scipy from https://files.pythonhosted.org/packages/df/64/ff35eb9e54894cf471ff4716abd3c81eb0a0626869217ce3e6ba4ccf17d7/scipy-1.18.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 12.0 MB/s eta 0:00:00:00:0100:01

[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


<hr />

## Atividade 1
Escreva os métodos de ponto fixo dentro do arquivo $utils/algoritmos.py$.

In [ ]:
import numpy as np
from scipy.optimize import approx_fprime

def pontofixo(a, g, TOL=1e-8):
    x = g(a)
    while abs(x - a) > TOL:
        a = x
        x = g(a)
    return x

def newton_raphson(a, f, TOL=1e-8, df=None):
    if df is None:
        def dfn(x):
            return approx_fprime(np.array([x]), lambda v: f(v[0]))[0]
    else:
        dfn = df
    g = lambda x: x - f(x) / dfn(x)
    return pontofixo(a, g, TOL)

def secante(a, b, f, TOL=1e-8):
    g = lambda a, b: (a * f(b) - b * f(a)) / (f(b) - f(a))
    x = g(a, b)
    while abs(x - b) > TOL:
        a, b = b, x
        x = g(a, b)
    return x

Resolver a equação $e^x = x + 2$ é equivalente a calcular os pontos fixos da função  

$$
g(x) = e^x - 2
$$

Use os métodos do ponto fixo  

$$
x^{(n+1)} = g(x^{(n)})
$$

com $x^{(0)} = -1.8$ para obter uma aproximação de uma das soluções da equação dada com 8 dígitos significativos.  

**Resposta:**  
$
x \approx -1.8414057
$

In [5]:
import numpy as np

import sys
import os

# Sobe um nível para a raiz do projeto e adiciona ao caminho do sistema
sys.path.append(os.path.abspath(os.path.join('..')))

from utils.algoritmos import (
    pontofixo,
    newton_raphson,
    secante,
)

f1 = lambda x: np.e**x - x - 2
g1 = lambda x: np.e**x - 2


def main():
    r = pontofixo(-1.8, g1)
    print(f"raiz ponto fixo = {r}")
    r = newton_raphson(-1.8, f1, df=lambda x: np.e**x - 1)
    print(f"raiz newton-raphson df = {r}")
    r = newton_raphson(-1.8, f1)
    print(f"raiz newton-raphson = {r}")
    r = secante(-1.8, -1.7, f1)
    print(f"raiz secante = {r}")


if __name__ == "__main__":
    main()

raiz ponto fixo = -1.841405660009702
raiz newton-raphson df = -1.8414056604369606
raiz newton-raphson = -1.8414056604369609
raiz secante = -1.841405660436961


## Atividade 2
Encontre a raiz positiva da função  

$$
f(x) = \cos(x) - x^2
$$  

pelos métodos do ponto fixo, inicializando-o com $x^{(0)} = 1$.  

Realize a iteração até obter estabilidade no **quinto dígito significativo**.  

**Resposta:**

$$
x \approx 0.82413 
$$

Processo iterativo:  

$$
x^{(n+1)} = x^{(n)} + \frac{\cos(x) - x^2}{\sin(x) + 2x}
$$

In [1]:
import math


def fixed_point_step(x_value: float) -> float:
    numerator = math.cos(x_value) - x_value**2
    denominator = math.sin(x_value) + 2 * x_value
    return x_value + numerator / denominator


x_value = 1.0
print(f"x^(0) = {x_value:.10f}")

for iteration in range(1, 101):
    next_x = fixed_point_step(x_value)
    print(f"x^({iteration}) = {next_x:.10f}")

    if format(next_x, ".5g") == format(x_value, ".5g"):
        x_value = next_x
        break

    x_value = next_x
else:
    raise RuntimeError("A iteração não estabilizou em cinco algarismos significativos.")

assert format(x_value, ".5g") == "0.82413"
print(f"Raiz aproximada: x ≈ {x_value:.5g}")

x^(0) = 1.0000000000
x^(1) = 0.8382184099
x^(2) = 0.8242418682
x^(3) = 0.8241323191
x^(4) = 0.8241323123
Raiz aproximada: x ≈ 0.82413


## Atividade 3
Aplique os métodos do ponto fixo para resolver a equação:

$$
e^{-x^2} = 2x
$$

**Resposta:**

$$
x \approx 0.4193648
$$

In [2]:
import math


def function(x_value: float) -> float:
    return math.exp(-(x_value**2)) - 2 * x_value


def fixed_point_step(x_value: float) -> float:
    return 0.5 * math.exp(-(x_value**2))


def fixed_point(initial: float, tolerance: float = 1e-12) -> tuple[float, int]:
    x_value = initial
    for iteration in range(1, 101):
        next_x = fixed_point_step(x_value)
        if abs(next_x - x_value) < tolerance:
            return next_x, iteration
        x_value = next_x
    raise RuntimeError("A iteração de ponto fixo não convergiu.")


def newton_raphson(initial: float, tolerance: float = 1e-12) -> tuple[float, int]:
    x_value = initial
    for iteration in range(1, 101):
        derivative = -2 * x_value * math.exp(-(x_value**2)) - 2
        next_x = x_value - function(x_value) / derivative
        if abs(next_x - x_value) < tolerance:
            return next_x, iteration
        x_value = next_x
    raise RuntimeError("Newton-Raphson não convergiu.")


def secant(first: float, second: float, tolerance: float = 1e-12) -> tuple[float, int]:
    value_first = function(first)
    value_second = function(second)

    for iteration in range(1, 101):
        denominator = value_second - value_first
        if denominator == 0.0:
            raise ZeroDivisionError("A secante encontrou uma divisão por zero.")

        next_x = second - value_second * (second - first) / denominator
        if abs(next_x - second) < tolerance:
            return next_x, iteration

        first, second = second, next_x
        value_first, value_second = value_second, function(next_x)

    raise RuntimeError("O método da secante não convergiu.")


methods = [
    ("Ponto fixo", fixed_point(0.5)),
    ("Newton-Raphson", newton_raphson(0.5)),
    ("Secante", secant(0.0, 0.5)),
]

for method_name, (root, iterations) in methods:
    assert abs(function(root)) < 1e-11
    assert math.isclose(root, 0.4193648, abs_tol=5e-8)
    print(f"{method_name}: x ≈ {root:.10f} ({iterations} iterações)")

Ponto fixo: x ≈ 0.4193648240 (26 iterações)
Newton-Raphson: x ≈ 0.4193648240 (4 iterações)
Secante: x ≈ 0.4193648240 (6 iterações)


## Atividade 4
Resolva os três últimos exercícios da semana anterior usando os métodos do ponto fixo.

In [ ]:
import math


ITERATION_TOLERANCE = 1e-11
MAX_ITERATIONS = 10_000


def fixed_point_iteration(mapping, initial: float) -> tuple[float, int]:
    current = initial
    for iteration in range(1, MAX_ITERATIONS + 1):
        next_value = mapping(current)
        if abs(next_value - current) <= ITERATION_TOLERANCE * max(1.0, abs(next_value)):
            return next_value, iteration
        current = next_value
    raise RuntimeError("A iteração de ponto fixo não convergiu.")


def newton_raphson(function, derivative, initial: float) -> tuple[float, int]:
    current = initial
    for iteration in range(1, 1_001):
        derivative_value = derivative(current)
        if derivative_value == 0.0:
            raise ZeroDivisionError("A derivada se anulou durante Newton-Raphson.")
        next_value = current - function(current) / derivative_value
        if abs(next_value - current) <= ITERATION_TOLERANCE * max(1.0, abs(next_value)):
            return next_value, iteration
        current = next_value
    raise RuntimeError("Newton-Raphson não convergiu.")


def secant(function, first: float, second: float) -> tuple[float, int]:
    first_value = function(first)
    second_value = function(second)

    for iteration in range(1, 1_001):
        denominator = second_value - first_value
        if denominator == 0.0:
            raise ZeroDivisionError("A secante encontrou uma divisão por zero.")
        next_value = second - second_value * (second - first) / denominator
        if abs(next_value - second) <= ITERATION_TOLERANCE * max(1.0, abs(next_value)):
            return next_value, iteration
        first, second = second, next_value
        first_value, second_value = second_value, function(next_value)
    raise RuntimeError("O método da secante não convergiu.")


def solve_with_methods(function, derivative, mapping, fixed_initial, newton_initial, secant_initials):
    return {
        "Ponto fixo": fixed_point_iteration(mapping, fixed_initial),
        "Newton-Raphson": newton_raphson(function, derivative, newton_initial),
        "Secante": secant(function, *secant_initials),
    }


BOLTZMANN_CONSTANT = 1.380649e-23
ELEMENTARY_CHARGE = 1.602176634e-19
TEMPERATURE = 300.0
REVERSE_CURRENT = 1e-12
THERMAL_VOLTAGE = BOLTZMANN_CONSTANT * TEMPERATURE / ELEMENTARY_CHARGE


def diode_residual(diode_voltage: float, source_voltage: float, resistance: float) -> float:
    current = REVERSE_CURRENT * math.expm1(diode_voltage / THERMAL_VOLTAGE)
    return resistance * current + diode_voltage - source_voltage


def diode_derivative(diode_voltage: float, resistance: float) -> float:
    return 1.0 + resistance * REVERSE_CURRENT / THERMAL_VOLTAGE * math.exp(
        diode_voltage / THERMAL_VOLTAGE
    )


def diode_fixed_map(diode_voltage: float, source_voltage: float, resistance: float) -> float:
    if source_voltage >= 1.0:
        log_argument = 1.0 + (source_voltage - diode_voltage) / (resistance * REVERSE_CURRENT)
        if log_argument <= 0.0:
            raise ValueError("A transformação logarítmica saiu do domínio real.")
        return THERMAL_VOLTAGE * math.log(log_argument)
    return source_voltage - resistance * REVERSE_CURRENT * math.expm1(
        diode_voltage / THERMAL_VOLTAGE
    )


diode_cases = [
    ("a", 30.0, 1e3, 0.623),
    ("b", 3.0, 1e3, 0.559),
    ("c", 3.0, 10e3, 0.500),
    ("d", 0.3, 1e3, 0.300),
    ("e", -0.3, 1e3, -0.300),
    ("f", -30.0, 1e3, -30.0),
    ("g", -30.0, 10e3, -30.0),
]

print("Circuito com diodo: v_t = {:.8f} V".format(THERMAL_VOLTAGE))
for label, source_voltage, resistance, expected_voltage in diode_cases:
    if source_voltage >= 1.0:
        initial_voltage = THERMAL_VOLTAGE * math.log1p(
            source_voltage / (resistance * REVERSE_CURRENT)
        )
        secant_initials = (0.0, initial_voltage)
    elif source_voltage > 0.0:
        initial_voltage = source_voltage
        secant_initials = (0.0, source_voltage)
    else:
        initial_voltage = source_voltage
        secant_initials = (source_voltage, 0.0)

    residual = lambda voltage: diode_residual(voltage, source_voltage, resistance)
    derivative = lambda voltage: diode_derivative(voltage, resistance)
    mapping = lambda voltage: diode_fixed_map(voltage, source_voltage, resistance)
    results = solve_with_methods(
        residual,
        derivative,
        mapping,
        fixed_initial=0.0,
        newton_initial=initial_voltage,
        secant_initials=secant_initials,
    )

    print(f"Caso {label}: V={source_voltage:g} V, R={resistance / 1000:g} kΩ")
    for method_name, (root, iterations) in results.items():
        assert abs(residual(root)) < 1e-6
        assert abs(root - expected_voltage) < 1e-3
        print(f"  {method_name}: v_d ≈ {root:.6g} V ({iterations} iterações)")


SPAN = 500.0
MAXIMUM_SAG = 50.0


def catenary_sag(parameter: float) -> float:
    return 2.0 * parameter * math.sinh(SPAN / (4.0 * parameter)) ** 2


def catenary_residual(parameter: float) -> float:
    return catenary_sag(parameter) - MAXIMUM_SAG


def catenary_derivative(parameter: float) -> float:
    scaled_span = SPAN / (2.0 * parameter)
    return math.cosh(scaled_span) - 1.0 - scaled_span * math.sinh(scaled_span)


catenary_mapping = lambda parameter: parameter + 10.0 * catenary_residual(parameter)
catenary_results = solve_with_methods(
    catenary_residual,
    catenary_derivative,
    catenary_mapping,
    fixed_initial=750.0,
    newton_initial=750.0,
    secant_initials=(500.0, 1_000.0),
)

print("\nCatenária:")
for method_name, (root, iterations) in catenary_results.items():
    assert abs(catenary_residual(root)) < 1e-7
    assert abs(root - 633.16218) < 1e-3
    print(f"  {method_name}: C ≈ {root:.6f} m, flecha ≈ {catenary_sag(root):.8f} m ({iterations} iterações)")


FREQUENCY = 1_000.0
INDUCTANCE = 0.1
LOAD_RESISTANCE = 1_000.0
TANGENT_PHI = 2.0 * math.pi * FREQUENCY * INDUCTANCE / LOAD_RESISTANCE
PHI = math.atan(TANGENT_PHI)


def rectifier_residual(beta: float) -> float:
    return math.sin(beta - PHI) + math.sin(PHI) * math.exp(-beta / TANGENT_PHI)


def rectifier_derivative(beta: float) -> float:
    return math.cos(beta - PHI) - math.sin(PHI) / TANGENT_PHI * math.exp(
        -beta / TANGENT_PHI
    )


def rectifier_mapping(beta: float) -> float:
    arcsine_argument = math.sin(PHI) * math.exp(-beta / TANGENT_PHI)
    return PHI + math.pi + math.asin(arcsine_argument)


rectifier_results = solve_with_methods(
    rectifier_residual,
    rectifier_derivative,
    rectifier_mapping,
    fixed_initial=math.pi,
    newton_initial=3.7,
    secant_initials=(3.5, 4.0),
)

print("\nRetificador:")
for method_name, (root, iterations) in rectifier_results.items():
    assert abs(rectifier_residual(root)) < 1e-10
    assert abs(math.degrees(root) - 212.2258175) < 1e-5
    print(f"  {method_name}: beta ≈ {math.degrees(root):.7f}° ({iterations} iterações)")

print("\nA resposta 212.2284° impressa na Semana 4 difere da raiz do modelo por cerca de 0.0026°.")

Circuito com diodo: v_t = 0.02585200 V
Caso a: V=30 V, R=1 kΩ
  Ponto fixo: v_d ≈ 0.623123 V (5 iterações)
  Newton-Raphson: v_d ≈ 0.623123 V (4 iterações)
  Secante: v_d ≈ 0.623123 V (7 iterações)
Caso b: V=3 V, R=1 kΩ
  Ponto fixo: v_d ≈ 0.55881 V (7 iterações)
  Newton-Raphson: v_d ≈ 0.55881 V (5 iterações)
  Secante: v_d ≈ 0.55881 V (11 iterações)
Caso c: V=3 V, R=10 kΩ
  Ponto fixo: v_d ≈ 0.4999 V (7 iterações)
  Newton-Raphson: v_d ≈ 0.4999 V (5 iterações)
  Secante: v_d ≈ 0.4999 V (10 iterações)
Caso d: V=0.3 V, R=1 kΩ
  Ponto fixo: v_d ≈ 0.299891 V (5 iterações)
  Newton-Raphson: v_d ≈ 0.299891 V (3 iterações)
  Secante: v_d ≈ 0.299891 V (3 iterações)
Caso e: V=-0.3 V, R=1 kΩ
  Ponto fixo: v_d ≈ -0.3 V (3 iterações)
  Newton-Raphson: v_d ≈ -0.3 V (2 iterações)
  Secante: v_d ≈ -0.3 V (2 iterações)
Caso f: V=-30 V, R=1 kΩ
  Ponto fixo: v_d ≈ -30 V (3 iterações)
  Newton-Raphson: v_d ≈ -30 V (2 iterações)
  Secante: v_d ≈ -30 V (2 iterações)
Caso g: V=-30 V, R=10 kΩ
  Ponto fixo:

## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 5"
git push origin semana5
```